# 期权希腊字母敏感性分析

可视化 Delta、Gamma、Theta、Vega 随标的价格和到期时间的变化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

def bs_greeks(S, K, T, r, sigma, option='call'):
    if T <= 0: return {'price':0,'delta':0,'gamma':0,'theta':0,'vega':0}
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if option == 'call':
        price = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
        delta = norm.cdf(d1)
    else:
        price = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
        delta = norm.cdf(d1) - 1
    gamma = norm.pdf(d1) / (S*sigma*np.sqrt(T))
    theta = (-(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)) / 365
    vega  = S*norm.pdf(d1)*np.sqrt(T) / 100
    return {'price':price,'delta':delta,'gamma':gamma,'theta':theta,'vega':vega}

K, T, r, sigma = 100, 0.25, 0.05, 0.2
S_range = np.linspace(70, 130, 200)

call_greeks = [bs_greeks(s, K, T, r, sigma, 'call') for s in S_range]
put_greeks  = [bs_greeks(s, K, T, r, sigma, 'put')  for s in S_range]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
pairs = [('price','期权价格'),('delta','Delta'),('gamma','Gamma'),
         ('theta','Theta (日)'),('vega','Vega (1% vol)')]

for idx, (key, label) in enumerate(pairs):
    ax = axes[idx//3][idx%3]
    ax.plot(S_range, [g[key] for g in call_greeks], 'b-', label='Call', lw=2)
    ax.plot(S_range, [g[key] for g in put_greeks],  'r-', label='Put',  lw=2)
    ax.axvline(K, color='gray', linestyle='--', alpha=0.5, label=f'K={K}')
    ax.set_title(label); ax.set_xlabel('标的价格'); ax.legend(); ax.grid(alpha=0.3)

axes[1][2].axis('off')
plt.suptitle(f'期权希腊字母 (K={K}, T={T}年, σ={sigma}, r={r})', fontsize=14)
plt.tight_layout()
plt.show()

## Theta 时间衰减

In [ ]:
T_range = np.linspace(0.01, 1.0, 100)
S = 100  # ATM
prices_atm = [bs_greeks(S, K, t, r, sigma)['price'] for t in T_range]
prices_itm = [bs_greeks(110, K, t, r, sigma)['price'] for t in T_range]
prices_otm = [bs_greeks(90,  K, t, r, sigma)['price'] for t in T_range]

plt.figure(figsize=(8, 4))
plt.plot(T_range, prices_atm, 'b-', label='平值 ATM (S=100)', lw=2)
plt.plot(T_range, prices_itm, 'g-', label='实值 ITM (S=110)', lw=2)
plt.plot(T_range, prices_otm, 'r-', label='虚值 OTM (S=90)',  lw=2)
plt.xlabel('到期时间 (年)')
plt.ylabel('Call 期权价格')
plt.title('时间价值衰减（Theta 效应）')
plt.legend(); plt.grid(alpha=0.3)
plt.show()